In [ ]:
import torch
import numpy as np
import librosa
import matplotlib.pyplot as plt
import IPython.display as ipd
from IPython.display import display, HTML

from sincnet import SincNet
from sincnet.mulaw import MuLawQuant

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
SAMPLE_RATE = 16_000
wav_np, _ = librosa.load("audio/invertibility/p232_001.wav", sr=SAMPLE_RATE, mono=True)
wav_np = wav_np / (np.abs(wav_np).max() + 1e-8)  # peak normalize
wav = torch.from_numpy(wav_np).unsqueeze(0).to(device)  # (1, T)
print(f"waveform: {wav.shape},  duration: {wav.shape[-1] / SAMPLE_RATE:.2f}s")
display(ipd.Audio(wav_np, rate=SAMPLE_RATE))

In [ ]:
sinc = SincNet(
    fs=SAMPLE_RATE,
    fps=128,
    n_bins=256,
    scale="mel",
    component="complex",
    causal=False,
    decoder_type="exact",
).eval().to(device)
print(sinc.config)

## 1 - Invertibility

In [ ]:
with torch.no_grad():
    spec  = sinc.encode(wav)                         # (1, 2, F, T)
    recon = sinc.decode(spec, length=wav.shape[-1])  # (1, T)

snr = 10 * torch.log10(
    wav.pow(2).sum() / (wav - recon).pow(2).sum()
).item()
print(f"SNR (exact decoder): {snr:.1f} dB")

recon_np = recon.squeeze(0).cpu().numpy()
display(HTML(f"""
<div style="display:flex; gap:40px; align-items:center;">
  <div><b>Original</b><br>{ipd.Audio(wav_np, rate=SAMPLE_RATE)._repr_html_()}</div>
  <div><b>Reconstructed ({snr:.1f} dB)</b><br>{ipd.Audio(recon_np, rate=SAMPLE_RATE)._repr_html_()}</div>
</div>
"""))

In [ ]:
mag = sinc.magnitude(spec)[0].cpu().numpy()  # (F, T)

fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(np.log1p(mag), origin="lower", aspect="auto", cmap="inferno")
ax.set_title("iSincNet magnitude spectrogram (mel-256, exact decoder)")
ax.set_xlabel("Frame")
ax.set_ylabel("Bin")
plt.tight_layout()
plt.show()

In [ ]:
N_FFT, HOP = 1024, 256

def stft_mag(x: torch.Tensor) -> np.ndarray:
    S = torch.stft(
        x.squeeze(0), n_fft=N_FFT, hop_length=HOP, win_length=N_FFT,
        window=torch.hann_window(N_FFT, device=x.device),
        return_complex=True,
    )
    return S.abs().cpu().numpy()

S_orig  = stft_mag(wav)
S_recon = stft_mag(recon)
vmax    = float(np.log1p(S_orig).max())

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
kw = dict(origin="lower", aspect="auto", cmap="magma", vmin=0, vmax=vmax)
axes[0].imshow(np.log1p(S_orig),  **kw)
axes[0].set_title("Original")
axes[1].imshow(np.log1p(S_recon), **kw)
axes[1].set_title(f"Reconstructed  ({snr:.1f} dB)")
for ax in axes:
    ax.set_xlabel("Frame")
    ax.set_ylabel("Freq bin")
plt.suptitle("STFT comparison: original vs iSincNet encode -> exact decode")
plt.tight_layout()
plt.show()

## 2 - Mu-law quantization

In [ ]:
Q_BITS = 8
quantizer = MuLawQuant(Q_BITS)

with torch.no_grad():
    q, scale  = quantizer.quantize(spec)
    spec_dq   = quantizer.dequantize(q, scale)
    recon_q   = sinc.decode(spec_dq, length=wav.shape[-1])

snr_q = 10 * torch.log10(
    wav.pow(2).sum() / (wav - recon_q).pow(2).sum()
).item()
print(f"SNR after {Q_BITS}-bit mu-law quantization: {snr_q:.1f} dB")

recon_q_np = recon_q.squeeze(0).cpu().numpy()
display(HTML(f"""
<div style="display:flex; gap:40px; align-items:center;">
  <div><b>Original</b><br>{ipd.Audio(wav_np, rate=SAMPLE_RATE)._repr_html_()}</div>
  <div><b>After {Q_BITS}-bit mu-law ({snr_q:.1f} dB)</b><br>{ipd.Audio(recon_q_np, rate=SAMPLE_RATE)._repr_html_()}</div>
</div>
"""))

## 3 - Linearity

In [ ]:
def load_wav(path: str) -> torch.Tensor:
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y = y / (np.abs(y).max() + 1e-8)
    return torch.from_numpy(y).unsqueeze(0).to(device)

stem_a = load_wav("audio/stems/stem_pad.wav")
stem_b = load_wav("audio/stems/stem_drum1.wav")

n = min(stem_a.shape[-1], stem_b.shape[-1])
stem_a, stem_b = stem_a[..., :n], stem_b[..., :n]

with torch.no_grad():
    mix_direct = sinc.encode(stem_a + stem_b)
    mix_linear = sinc.encode(stem_a) + sinc.encode(stem_b)

err = (mix_direct - mix_linear).abs().max().item()
print(f"max |encode(a+b) - (encode(a)+encode(b))| = {err:.2e}  (machine precision -> linear)")